# Perceptual MM-Cache Benchmark — Colab Pro

GPU-heavy execution layer for the `pmcache` project. This notebook:
1. Spins up Qwen2-VL-2B + vLLM + LMCache
2. Runs the **baseline** (LMCache bytewise mm_hash only)
3. Runs the **perceptual** variant (pmcache enabled)
4. Sweeps the similarity threshold τ
5. Generates a final report with plots + tables
6. (Optional) Launches a live Gradio demo with public share link

**Before running:** `Runtime → Change runtime type → A100` (preferred) or `L4` (24 GB, also fine). T4 (16 GB) is too small for Qwen2-VL with LMCache headroom.

**Prereq:** the `perceptual-mmcache` repo must be pushed to GitHub. Set `REPO_URL` in the cell below.

In [ ]:
# ====== EDIT THIS ======
REPO_URL = "https://github.com/YOUR_USERNAME/perceptual-mmcache.git"
BRANCH = "main"
MODEL = "Qwen/Qwen2-VL-2B-Instruct"
# =======================

import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

## 0. GPU sanity check

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "No GPU detected — change runtime type to A100/L4"
props = torch.cuda.get_device_properties(0)
vram_gb = props.total_memory / 1e9
print(f"\nDevice: {props.name}")
print(f"VRAM:   {vram_gb:.1f} GB")
assert vram_gb >= 20, f"Need ≥20 GB VRAM for Qwen2-VL-2B + LMCache. Got {vram_gb:.1f} GB."

## 1. Workspace on Google Drive

All results and downloaded videos persist to Drive so a disconnected session doesn't lose progress.

In [ ]:
# --- Workspace setup ----------------------------------------------------
# Set the workspace once via PMCACHE_WORKSPACE; eval.paths (imported in
# the post-clone cell) reads it and exposes VIDEOS_DIR / RESULTS_DIR /
# SWEEP_DIR / REPORT_DIR / BASELINE_PATH / PERCEPTUAL_PATH consistently.
# To benchmark against a different video set, change WORKSPACE below
# (or export PMCACHE_WORKSPACE before launching Colab).
from google.colab import drive
drive.mount('/content/drive')

import os
WORKSPACE = '/content/drive/MyDrive/pmcache'
os.environ['PMCACHE_WORKSPACE'] = WORKSPACE

# Plain shortcuts for the pre-clone cells that can't yet import eval.paths.
BASELINE_PATH   = f'{WORKSPACE}/results/baseline.jsonl'
PERCEPTUAL_PATH = f'{WORKSPACE}/results/perceptual.jsonl'
SWEEP_DIR       = f'{WORKSPACE}/results/sweep'
REPORT_DIR      = f'{WORKSPACE}/results/report'
VIDEOS_DIR      = f'{WORKSPACE}/videos'

for d in (VIDEOS_DIR, SWEEP_DIR, REPORT_DIR):
    os.makedirs(d, exist_ok=True)

print(f"Workspace: {WORKSPACE}")

## 2. Install dependencies

vLLM + LMCache install can take a few minutes on first run. Re-running this cell after a session restart is fast (cached wheels).

In [ ]:
%%capture
!pip install -q --upgrade pip
!pip install -q vllm
!pip install -q lmcache
!pip install -q imagehash pybktree
!pip install -q opencv-python-headless pillow
!pip install -q transformers accelerate hf_transfer
!pip install -q matplotlib pandas tqdm seaborn gradio pytest

In [ ]:
# Verify versions
import vllm, lmcache, torch, transformers
print(f"vllm:         {vllm.__version__}")
print(f"lmcache:      {lmcache.__version__ if hasattr(lmcache, '__version__') else '(installed)'}")
print(f"torch:        {torch.__version__}")
print(f"transformers: {transformers.__version__}")

## 3. Clone project repo + install

In [ ]:
%cd /content
!rm -rf perceptual-mmcache
!git clone -b {BRANCH} {REPO_URL}
%cd /content/perceptual-mmcache
!pip install -q -e .

In [ ]:
# Re-import paths now that the repo is on sys.path so subsequent cells
# can refer to eval.paths.VIDEOS_DIR etc. instead of the f-string shortcuts.
from eval import paths
paths.set_workspace(WORKSPACE)
paths.ensure_dirs()
print(f"eval.paths.VIDEOS_DIR  = {paths.VIDEOS_DIR}")
print(f"eval.paths.QA_FILE     = {paths.QA_FILE}")
print(f"eval.paths.RESULTS_DIR = {paths.RESULTS_DIR}")

# Run unit tests as a sanity check (these should pass without GPU).
!pytest tests/ -q --tb=short

## 4. Prepare test dataset

Downloads ~8 short CC0 clips (static talking-head, slideshow, screen recording, low-motion outdoor) and writes `qa.jsonl` with hand-written questions and gold answers. Skipped if videos already exist on Drive.

In [ ]:
import os
video_files = [f for f in os.listdir(paths.VIDEOS_DIR)
               if f.endswith(('.mp4', '.mov', '.webm'))]
if len(video_files) < 5:
    # Default: synth mode generates 5 deterministic cache-stress videos
    # offline. To use real CC0 clips, pass --mode manifest --manifest <path>.
    !python -m eval.datasets.prepare_demo_videos --output_dir {paths.VIDEOS_DIR}
else:
    print(f"Already have {len(video_files)} videos in {paths.VIDEOS_DIR}, skipping.")

video_files = sorted(f for f in os.listdir(paths.VIDEOS_DIR)
                     if f.endswith(('.mp4', '.mov', '.webm')))
print(f"\nVideos ({len(video_files)}):")
for v in video_files:
    print(f"  - {v}")

## 5. VLM smoke test

Loads Qwen2-VL-2B once and runs a single image inference to verify the stack is healthy before committing to long benchmark runs. First run downloads ~5 GB of weights.

In [ ]:
from eval.utils import smoke_test_vlm
smoke_test_vlm(model=MODEL, workspace=WORKSPACE)

## 6. Baseline benchmark

vLLM + LMCache with bytewise `mm_hash` only — what every video-LLM deployment ships today. Expected: ~0% cross-frame cache hit rate.

In [ ]:
from eval.run_baseline import run_baseline_benchmark

run_baseline_benchmark(
    videos_dir=paths.VIDEOS_DIR,
    qa_file=paths.QA_FILE,
    output_path=paths.BASELINE_PATH,
    model=MODEL,
    fps=1.0,
)

print(f"\n✓ Baseline saved to: {paths.BASELINE_PATH}")

## 7. Perceptual benchmark

Same setup, but with pmcache enabled. Expected: high cross-frame hit rate on low-motion content; TTFT collapses on second+ frames.

In [ ]:
from eval.run_perceptual import run_perceptual_benchmark

run_perceptual_benchmark(
    videos_dir=paths.VIDEOS_DIR,
    qa_file=paths.QA_FILE,
    output_path=paths.PERCEPTUAL_PATH,
    model=MODEL,
    tau=0.98,
    k=5,
    fps=1.0,
)

print(f"\n✓ Perceptual saved to: {paths.PERCEPTUAL_PATH}")

## 8. Threshold sweep

Sweeps cosine threshold τ to find the operating point where accuracy is preserved and hit rate is maximized.

In [ ]:
from eval.threshold_sweep import sweep_thresholds

sweep_thresholds(
    videos_dir=paths.VIDEOS_DIR,
    qa_file=paths.QA_FILE,
    taus=[0.95, 0.96, 0.97, 0.98, 0.99],
    output_dir=paths.SWEEP_DIR,
    model=MODEL,
    baseline_path=paths.BASELINE_PATH,
)

print(f"\n✓ Sweep results saved to: {paths.SWEEP_DIR}")

## 9. Final report

Pulls baseline + perceptual + sweep data, computes headline metrics, generates plots, writes `REPORT.md`.

In [ ]:
from eval.benchmark_videoqa import analyze_results

analyze_results(
    baseline_path=paths.BASELINE_PATH,
    perceptual_path=paths.PERCEPTUAL_PATH,
    sweep_dir=paths.SWEEP_DIR,
    output_dir=paths.REPORT_DIR,
)

print(f"\n✓ Report saved to: {paths.REPORT_DIR}/REPORT.md")

In [ ]:
# Display report inline
from IPython.display import Markdown, display
with open(f'{REPORT_DIR}/REPORT.md') as f:
    display(Markdown(f.read()))

In [ ]:
# Display the plot images
from IPython.display import Image
import os

for png in sorted(os.listdir(REPORT_DIR)):
    if png.endswith('.png'):
        print(f"\n=== {png} ===")
        display(Image(filename=f'{REPORT_DIR}/{png}'))

## 10. (Optional) Live demo with public link

Launches a Gradio app inside Colab and exposes a public share URL via Gradio's tunnel. Use this for the hackathon presentation — judges click the link and try it live.

In [ ]:
from demo.gradio_app import launch_demo

launch_demo(
    model=MODEL,
    workspace=WORKSPACE,
    share=True,
)

## 11. Save artifacts back to Drive

Most things already write directly to Drive, but use this as a final sync before disconnecting.

In [ ]:
!ls -lh {WORKSPACE}/results/
print()
!ls -lh {REPORT_DIR}/

## Troubleshooting

**`AssertionError: Need ≥20 GB VRAM`** — Runtime is on T4. Change to A100 or L4: `Runtime → Change runtime type → A100 GPU`.

**vLLM OOM during model load** — Lower `gpu_memory_utilization` in `eval/utils.py` (default 0.85, try 0.75). Or restart the runtime to free VRAM and re-run from the install cell.

**LMCache connector errors** — Check `lmcache` version. Pin a working version in `pyproject.toml` if needed; the multimodal mm_hash path was added in 0.3.1.

**`mm_hash aliasing doesn't fire`** — Verify `PMCACHE_ENABLED=true` is set in the perceptual run, and that the shim subclass actually loads (add a print in `lmcache_shim.py` on import).

**Accuracy regression > 1pp in perceptual run** — τ is too loose. Re-run the threshold sweep, pick a τ where accuracy is on the plateau.

**Session disconnected mid-benchmark** — Drive persisted everything up to the last completed JSONL write. Restart runtime, skip to the cell after the last completed step.

**Gradio share link doesn't work** — Colab sometimes blocks tunnels. Use `share=False` and the local `https://...gradio.live` link from the cell output, or pre-record the demo as a screen capture.